In [1]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
pip install pillow

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
# ===============================
# 1. IMPORTS
# ===============================
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import json
import os

# ===============================
# 2. CONFIG
# ===============================
DATASET_PATH = "garbage_classification"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ===============================
# 3. DATA LOADING
# ===============================
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ===============================
# 4. SAVE CLASS NAMES
# ===============================
os.makedirs("models", exist_ok=True)

class_names = list(train_data.class_indices.keys())

with open("models/classes.json", "w") as f:
    json.dump(class_names, f)

print("Classes:", class_names)

# ===============================
# 5. MODEL (TRANSFER LEARNING)
# ===============================
base_model = MobileNetV2(weights='imagenet', include_top=False)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
output = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ===============================
# 6. TRAIN
# ===============================
model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# ===============================
# 7. SAVE MODEL
# ===============================
model.save("models/model.h5")

print("✅ Model trained and saved!")

Found 12415 images belonging to 12 classes.
Found 3100 images belonging to 12 classes.
Classes: ['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']


C:\Users\DELL\AppData\Local\Temp\ipykernel_31552\2669670825.py:58: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False)


Epoch 1/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 331s 847ms/step - accuracy: 0.8881 - loss: 0.3730 - val_accuracy: 0.8890 - val_loss: 0.3380
Epoch 2/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 220s 568ms/step - accuracy: 0.9515 - loss: 0.1520 - val_accuracy: 0.8942 - val_loss: 0.3293
Epoch 3/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 205s 529ms/step - accuracy: 0.9714 - loss: 0.0907 - val_accuracy: 0.8965 - val_loss: 0.3123
Epoch 4/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 212s 547ms/step - accuracy: 0.9843 - loss: 0.0560 - val_accuracy: 0.9068 - val_loss: 0.3234
Epoch 5/5
388/388 ━━━━━━━━━━━━━━━━━━━━ 215s 554ms/step - accuracy: 0.9874 - loss: 0.0410 - val_accuracy: 0.9061 - val_loss: 0.3323


✅ Model trained and saved!
